In [ ]:
pip install geemap==0.32.0

In [ ]:
pip install tk==0.1.0

In [ ]:
!pip install pycrs

In [ ]:
import os
import ee
import geemap
from pathlib import Path
from datetime import datetime, timedelta , date
import time
import pandas as pd
import csv
import glob
import folium
from IPython.display import display, clear_output
import shutil
import shapefile
import ipywidgets as widgets
from tkinter import Tk, filedialog
import traitlets
from ipyfilechooser import FileChooser
import math
import logging

In [ ]:
class SelectFilesButton(widgets.Button):
    """A file widget that leverages tkinter.filedialog."""

    def __init__(self):
        super(SelectFilesButton, self).__init__()
        # Add the selected_files trait
        self.add_traits(files=traitlets.traitlets.List())
        # Create the button.
        self.description = "Select Files"
        self.icon = "square-o"
        self.style.button_color = "orange"
        # Set on click behavior.
        self.on_click(self.select_files)

    @staticmethod
    def select_files(b):
        """Generate instance of tkinter.filedialog.

        Parameters
        ----------
        b : obj:
            An instance of ipywidgets.widgets.Button 
        """
        # Create Tk root
        root = Tk()
        # Hide the main window
        root.withdraw()
        # Raise the root to the top of all windows.
        root.call('wm', 'attributes', '.', '-topmost', True)
        # List of selected fileswill be set to b.value
        b.files = filedialog.askopenfilename(multiple=True,filetypes=[('shp','.shp')])

        b.description = "Files Selected"
        b.icon = "check-square-o"
        b.style.button_color = "lightgreen"


def _last_day_of_month(any_day):
    next_month = any_day.replace(day=28) + timedelta(days=4)  # this will never fail
    return next_month - timedelta(days=next_month.day)


def monthlist(begin,end):
    # begin = datetime.strptime(begin, "%Y-%m-%d")
    # end = datetime.strptime(end, "%Y-%m-%d")

    result = []
    while True:
        if begin.month == 12:
            next_month = begin.replace(year=begin.year+1,month=1, day=1)
        else:
            next_month = begin.replace(month=begin.month+1, day=1)
        if next_month > end:
            break
        result.append ([begin.strftime("%Y-%m-%d"),_last_day_of_month(begin).strftime("%Y-%m-%d")])
        begin = next_month
    result.append ([begin.strftime("%Y-%m-%d"),end.strftime("%Y-%m-%d")])
    return result


def monthlist(begin,end):
    # begin = datetime.strptime(begin, "%Y-%m-%d")
    # end = datetime.strptime(end, "%Y-%m-%d")

    result = []
    while True:
        if begin.month == 12:
            next_month = begin.replace(year=begin.year+1,month=1, day=1)
        else:
            next_month = begin.replace(month=begin.month+1, day=1)
        if next_month > end:
            break
        result.append ([begin.strftime("%Y-%m-%d"),_last_day_of_month(begin).strftime("%Y-%m-%d")])
        begin = next_month
    result.append ([begin.strftime("%Y-%m-%d"),end.strftime("%Y-%m-%d")])
    return result


def date_format_concersion(date, output_format='%Y/%m/%d'):

    # Fool-proof: check if the input date is None
    if date is None:
        return None

    try: 
        parsed_date = datetime.strptime(date, output_format)
        
    except ValueError as e:

        try:
            # Try to parse the input date
            parsed_date = datetime.strptime(date, '%Y%m%d')
        except ValueError as e:
        
            try:
                parsed_date = datetime.strptime(date, '%Y_%m_%d')
            except ValueError as e:

                try:
                    parsed_date = datetime.strptime(date, '%Y-%m-%d')
                except ValueError as e:

                    return logging.error(f'Unparsable date format {e}')

    output_date = parsed_date.strftime(output_format)

    return output_date

def getRH(image):
    
    RH = image.expression(
      '100 * (exp((17.625 * Td)/(243.04 + Td))/exp((17.625*T)/(243.04 + T)))',{
      'Td' : image.select('dewpoint_temperature_2m').add(-273.15),
      'T' : image.select('temperature_2m').add(-273.15)}).rename('Relative_Humidity')
    
    return image.addBands(RH)

In [ ]:
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
# After executing this line of code for the first use, you can get the authentication number linked to Google.
Map = geemap.Map()
# Authenticate the Google earth engine with google account
ee.Initialize()

In [ ]:
# give the star date and end date

star = widgets.DatePicker(
    description='Pick a Star Date',
    disabled=False
)
end = widgets.DatePicker(
    description='Pick a End Date',
    disabled=False
)

widgets.HBox([star, end])

In [ ]:
import os

# Replace 'your_directory_path' with the actual path
directory_J = 'D:/EU_shp_separate'

# create an empty list to store the results
paths_list = []

# List all files in this directory
files_J = os.listdir(directory_J)

files_J 

In [ ]:
files_J [1]

In [ ]:
len(time_list)

In [ ]:
len(env_var_list)

In [ ]:
len(statics_OwO)

In [ ]:
shpfile_OwO

In [ ]:
original_list={
    'part1': ['dewpoint_temperature_2m', 'temperature_2m', 'temperature_2m_min', 'temperature_2m_max', 'skin_temperature'],
    'part2': ['soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature'],
    'part3': ['lake_ice_depth', 'lake_ice_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_shape_factor'],
    'part4': ['lake_total_layer_temperature', 'snow_albedo', 'snow_cover', 'snow_density', 'snow_depth'],
    'part5': ['snow_depth_water_equivalent', 'snowfall_sum', 'snowmelt_sum', 'temperature_of_snow_layer', 'skin_reservoir_content'],
    'part6': ['volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'forecast_albedo'],
    'part7': ['surface_latent_heat_flux_sum', 'surface_net_solar_radiation_sum', 'surface_net_thermal_radiation_sum', 'surface_sensible_heat_flux_sum', 'surface_solar_radiation_downwards_sum'],
    'part8': ['surface_thermal_radiation_downwards_sum', 'evaporation_from_bare_soil_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', 'evaporation_from_the_top_of_canopy_sum', 'evaporation_from_vegetation_transpiration_sum'],
    'part9': ['potential_evaporation_sum', 'potential_evaporation_min', 'potential_evaporation_max', 'runoff_sum', 'snow_evaporation_sum'],
    'part10': ['sub_surface_runoff_sum', 'surface_runoff_sum', 'total_evaporation_sum', 'total_evaporation_min', 'total_evaporation_max'],
    'part11': ['u_component_of_wind_10m', 'v_component_of_wind_10m', 'surface_pressure', 'total_precipitation_sum', 'total_precipitation_min'],
    'part12': ['total_precipitation_max', 'leaf_area_index_high_vegetation', 'leaf_area_index_low_vegetation', 'Relative_Humidity']
}


i=12
part_name = f'part{i}'
bandname = original_list[part_name]
bandname
part_name

In [ ]:
#J loop, but time need to select new

##list file
import os

# Replace 'your_directory_path' with the actual path
#directory_J = '/media/dyclab/新增磁碟區/Jeffery/data/shp/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp'
directory_J = "D:/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp"

#--------------------------------------------------------------
# list of environment variables
original_list={
    'part1': ['dewpoint_temperature_2m', 'temperature_2m', 'temperature_2m_min', 'temperature_2m_max', 'skin_temperature'],
    'part2': ['soil_temperature_level_1', 'soil_temperature_level_2', 'soil_temperature_level_3', 'soil_temperature_level_4', 'lake_bottom_temperature'],
    'part3': ['lake_ice_depth', 'lake_ice_temperature', 'lake_mix_layer_depth', 'lake_mix_layer_temperature', 'lake_shape_factor'],
    'part4': ['lake_total_layer_temperature', 'snow_albedo', 'snow_cover', 'snow_density', 'snow_depth'],
    'part5': ['snow_depth_water_equivalent', 'snowfall_sum', 'snowmelt_sum', 'temperature_of_snow_layer', 'skin_reservoir_content'],
    'part6': ['volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2', 'volumetric_soil_water_layer_3', 'volumetric_soil_water_layer_4', 'forecast_albedo'],
    'part7': ['surface_latent_heat_flux_sum', 'surface_net_solar_radiation_sum', 'surface_net_thermal_radiation_sum', 'surface_sensible_heat_flux_sum', 'surface_solar_radiation_downwards_sum'],
    'part8': ['surface_thermal_radiation_downwards_sum', 'evaporation_from_bare_soil_sum', 'evaporation_from_open_water_surfaces_excluding_oceans_sum', 'evaporation_from_the_top_of_canopy_sum', 'evaporation_from_vegetation_transpiration_sum'],
    'part9': ['potential_evaporation_sum', 'potential_evaporation_min', 'potential_evaporation_max', 'runoff_sum', 'snow_evaporation_sum'],
    'part10': ['sub_surface_runoff_sum', 'surface_runoff_sum', 'total_evaporation_sum', 'total_evaporation_min', 'total_evaporation_max'],
    'part11': ['u_component_of_wind_10m', 'v_component_of_wind_10m', 'surface_pressure', 'total_precipitation_sum', 'total_precipitation_min'],
    'part12': ['total_precipitation_max', 'leaf_area_index_high_vegetation', 'leaf_area_index_low_vegetation', 'Relative_Humidity']
}

#convert each element into a single-element tuple and store it in a new list
#env_var_list = [(item,) for item in original_list]
#band_name = ('temperature_2m',)

#--------------------------------------------------------------
# statics list
#statics_OwO =['MEAN','MAXIMUM', 'MINIMUM', 'MEDIAN', 'STD', 'VARIANCE', 'SUM']
#statics = 'MEAN'
statics_OwO =['MEDIAN']

#-----------------------------------------------
file_name = 'Id'

#---------------------------------------------------------

# create folder
folder_name = 'data_all'

# create folder name : data_all
if os.path.isdir(folder_name) == True:
    shutil.rmtree(folder_name, ignore_errors=True)
    os.makedirs(folder_name)
else:
    os.makedirs(folder_name)

#-------------------------------------------------------------

import time

#band_name = original_list
shpfile_OwO = directory_J

    
for part_number in range(7, 13):
    part_name = f'part{part_number}'
    band_name = original_list[part_name]

    start_time = time.time()
    time_list = monthlist(star.value, end.value)
    states = geemap.shp_to_ee(shpfile_OwO)

    for i in range(0, len(time_list)):
        for statics in statics_OwO[0:1]:  # Moved this loop outside of the previous indentation
            Era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
                .filter(ee.Filter.date(datetime.strptime(time_list[i][0], "%Y-%m-%d"), datetime.strptime(time_list[i][1], "%Y-%m-%d")+timedelta(days=1))) \
                .map(getRH) \
                .map(lambda image: image.select(list(band_name))) \
                .map(lambda image: image.clip(states)) \
                .map(lambda image: image.reproject(crs='EPSG:4326'))

            if Era5.size().getInfo() > 0:
                Era5 = Era5.toBands()
            else:
                print("No images found for the given date range.")
                continue
            
            out_dir = os.path.expanduser(folder_name)
            out_dem_stats = os.path.join(
                out_dir, 'Era5_{}_{}_{}.csv'.format(part_name, statics, time_list[i]))

            if not os.path.exists(out_dir):
                os.makedirs(out_dir)

            geemap.zonal_statistics(
                Era5, states, out_dem_stats, statistics_type=statics, scale=1000)

            data_temp = pd.read_csv(out_dem_stats)

            data = []

            if (len(band_name)) == 1 & (time_list[i][0] == time_list[i][1]):

                df = data_temp
                df[file_name] = data_temp.loc[:, [file_name]]
                df['Date'] = date_format_concersion(
                    time_list[i][0], output_format='%Y/%m/%d')
                df['Doy'] = datetime.strptime(
                    time_list[i][0], '%Y-%m-%d').strftime('%j')
                select_columns = ['Date', 'Doy'] + [item.lower() for item in [statics]] + [file_name]
                df = df[select_columns]
                new_columns = ['Date', 'Doy'] + band_name + [file_name]
                df.columns = new_columns

                data.append(df)

            else:
                column_name_list = data_temp.columns.tolist()
                c = []
                d = []
                for k in zip(column_name_list[:]):
                    c.append(k[0][0:8])
                    d.append(k[0])

                data = []
                for j in range(0, len(column_name_list), len(band_name)):

                    # Check date format and extract data Check date format and extract data
                    if all(m.isdigit() for m in c[j:j+len(band_name)]):

                        # Extract data
                        df = data_temp.loc[:, d[j:j+len(band_name)]]
                        df[file_name] = data_temp.loc[:, [file_name]]

                        # Create new date and DOY columns
                        df.insert(0, 'Date', '')
                        df['Date'] = date_format_concersion(
                            time_list[i][0], output_format='%Y/%m/%d')

                        df.insert(1, 'Doy', '')
                        df['Doy'] = datetime.strptime(
                            time_list[i][0], '%Y-%m-%d').strftime('%j')

                        # Rename columns
                        colnames = ['Date', 'Doy']
                        colnames.extend(list(band_name))
                        colnames.append(file_name)
                        df.columns = [colnames]

                        data.append(df)
                    else:
                        continue

            appended_data = pd.concat(data, axis=0, ignore_index=True)

            # Output the file with date and doy back
            appended_data.to_csv(out_dem_stats, index=False)

    end_time = time.time()
    # execution time of this loop
    elapsed_time = end_time - start_time
    print(f"迴圈執行時間: {elapsed_time:.4f} 秒")

    def _rename_columns(df, column_name, suffix):
        if column_name in df.columns:
            df.rename(columns={column_name: f"{column_name}_{suffix}"}, inplace=True)

    def cbind_era5(partname, statics):
        all_files = glob.glob(os.path.join(folder_name, "Era5_{}_{}*.csv".format(part_name, statics)))

        df_from_each_file = (pd.read_csv(f, sep=",") for f in all_files)
        df_merged = pd.concat(df_from_each_file, ignore_index=True)

        for column_name in df_merged.columns:
            if column_name in band_name:
                _rename_columns(df_merged, column_name, str(statics))

        band_name_str = ', '.join(band_name)

        df_merged.to_csv('D:/EU_result/EU_100km_result/' + 
                         part_name +
                         '/' +
                         statics +
                         '_' +
                         str(star.value) +
                         '_' +
                         'to' +
                         '_' +
                         str(end.value) +
                         '.csv', index=False)

    cbind_era5(part_name,statics)
    
    # after executing this line of code for the first use, you can get the authentication number linked to Google.
    Map = geemap.Map()
    # Authenticate the Google earth engine with google account
    ee.Initialize()